# Validação do projeto — ecommerce_lakehouse

Cada seção tem uma descrição curta e a consulta correspondente. Execute as células manualmente e tire o print do resultado junto com o título.

## 00 · Quem está executando

Mostra o usuário atual e se ele pertence a cada um dos três grupos do projeto, para saber com qual identidade as demais consultas rodam.

In [4]:
%sql
SELECT current_user() AS usuario,
       current_catalog() AS catalogo,
       is_account_group_member('ecom_engenharia')   AS engenharia,
       is_account_group_member('ecom_analistas')    AS analistas,
       is_account_group_member('ecom_pii_leitores') AS pii_leitores;

SyntaxError: invalid syntax (1283634731.py, line 2)

## 01a · Detalhes da tabela Delta

Exibe o formato, a localização, o número de arquivos e as versões de protocolo de leitura e escrita da tabela de auditoria.

In [ ]:
%sql
DESCRIBE DETAIL ecommerce_lakehouse.auditoria.pedidos_snapshot;

## 01b · Histórico de transações

Lista cada commit do log de transações da tabela, com a operação executada, seus parâmetros e suas métricas.

In [ ]:
%sql
SELECT version, timestamp, operation, operationParameters, operationMetrics
FROM (DESCRIBE HISTORY ecommerce_lakehouse.auditoria.pedidos_snapshot)
ORDER BY version;

## 01c · Propriedades da tabela

Mostra as propriedades e os recursos do Delta habilitados na tabela, como os vetores de exclusão.

In [ ]:
%sql
SHOW TBLPROPERTIES ecommerce_lakehouse.auditoria.pedidos_snapshot;

## 02a · Tabelas por camada

Lista as tabelas e views de cada schema com o seu tipo: streaming table, materialized view, view ou tabela comum.

In [ ]:
%sql
SELECT table_schema AS camada, table_name, table_type
FROM ecommerce_lakehouse.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'gold', 'referencia', 'auditoria')
  AND table_name NOT LIKE '__materialization%'
ORDER BY CASE table_schema WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 WHEN 'gold' THEN 3 ELSE 4 END,
         table_name;

## 02b · Linhas por tabela

Conta as linhas das principais tabelas de cada camada, mostrando o refinamento dos dados da bronze até a gold.

In [ ]:
%sql
SELECT 'bronze.clientes_raw' AS tabela, count(*) AS linhas FROM ecommerce_lakehouse.bronze.clientes_raw
UNION ALL SELECT 'bronze.pedidos_raw', count(*) FROM ecommerce_lakehouse.bronze.pedidos_raw
UNION ALL SELECT 'bronze.produtos_raw', count(*) FROM ecommerce_lakehouse.bronze.produtos_raw
UNION ALL SELECT 'silver.clientes_eventos_validos', count(*) FROM ecommerce_lakehouse.silver.clientes_eventos_validos
UNION ALL SELECT 'silver.pedidos_eventos_validos', count(*) FROM ecommerce_lakehouse.silver.pedidos_eventos_validos
UNION ALL SELECT 'silver.pedidos_quarentena', count(*) FROM ecommerce_lakehouse.silver.pedidos_quarentena
UNION ALL SELECT 'silver.clientes', count(*) FROM ecommerce_lakehouse.silver.clientes
UNION ALL SELECT 'silver.pedidos', count(*) FROM ecommerce_lakehouse.silver.pedidos
UNION ALL SELECT 'silver.produtos', count(*) FROM ecommerce_lakehouse.silver.produtos
UNION ALL SELECT 'gold.vendas_diarias_categoria', count(*) FROM ecommerce_lakehouse.gold.vendas_diarias_categoria
UNION ALL SELECT 'gold.funil_status_pedidos', count(*) FROM ecommerce_lakehouse.gold.funil_status_pedidos
UNION ALL SELECT 'gold.clientes_valor', count(*) FROM ecommerce_lakehouse.gold.clientes_valor
UNION ALL SELECT 'gold.dim_clientes', count(*) FROM ecommerce_lakehouse.gold.dim_clientes
UNION ALL SELECT 'gold.pedidos_por_hora', count(*) FROM ecommerce_lakehouse.gold.pedidos_por_hora;

## 03 · Execução dos flows do pipeline

Mostra, a partir do event log, o status final de cada flow do pipeline em cada atualização.

In [ ]:
%sql
SELECT timestamp,
       origin.update_id AS update_id,
       origin.flow_name AS flow,
       details:flow_progress.status::STRING AS status
FROM ecommerce_lakehouse.monitoramento.vw_event_log
WHERE event_type = 'flow_progress'
  AND details:flow_progress.status::STRING IN ('COMPLETED', 'FAILED')
ORDER BY timestamp DESC
LIMIT 40;

## 04a · Arquivos ingeridos pelo Auto Loader

Conta as linhas da bronze por arquivo de origem, mostrando que cada arquivo foi lido uma única vez.

In [ ]:
%sql
SELECT 'clientes_raw' AS tabela, arquivo_origem, count(*) AS linhas, min(data_ingestao) AS ingerido_em
FROM ecommerce_lakehouse.bronze.clientes_raw GROUP BY arquivo_origem
UNION ALL
SELECT 'pedidos_raw', arquivo_origem, count(*), min(data_ingestao)
FROM ecommerce_lakehouse.bronze.pedidos_raw GROUP BY arquivo_origem
UNION ALL
SELECT 'produtos_raw', arquivo_origem, count(*), min(data_ingestao)
FROM ecommerce_lakehouse.bronze.produtos_raw GROUP BY arquivo_origem
ORDER BY 1, 2;

## 04b · Dados fora do schema

Conta as linhas da bronze com a coluna `_rescued_data` preenchida, ou seja, com campos que não encaixaram no schema esperado.

In [ ]:
%sql
SELECT 'clientes_raw' AS tabela, count(*) AS com_rescued_data
FROM ecommerce_lakehouse.bronze.clientes_raw WHERE _rescued_data IS NOT NULL
UNION ALL
SELECT 'pedidos_raw', count(*) FROM ecommerce_lakehouse.bronze.pedidos_raw WHERE _rescued_data IS NOT NULL
UNION ALL
SELECT 'produtos_raw', count(*) FROM ecommerce_lakehouse.bronze.produtos_raw WHERE _rescued_data IS NOT NULL;

## 05a · Expectations

Soma os registros aprovados e reprovados de cada regra de qualidade definida no pipeline.

In [ ]:
%sql
SELECT dataset, regra, sum(aprovados) AS total_aprovados, sum(reprovados) AS total_reprovados
FROM ecommerce_lakehouse.monitoramento.vw_qualidade_expectations
GROUP BY ALL
ORDER BY dataset, regra;

## 05b · Quarentena por regra

Conta os pedidos rejeitados na quarentena, agrupados pela regra violada.

In [ ]:
%sql
SELECT regra, count(*) AS qtd
FROM ecommerce_lakehouse.silver.pedidos_quarentena
LATERAL VIEW explode(regras_violadas) AS regra
GROUP BY regra
ORDER BY qtd DESC;

## 05c · Amostra da quarentena

Mostra alguns pedidos rejeitados, com a lista de regras que cada um violou.

In [ ]:
%sql
SELECT pedido_id, cliente_id, quantidade, preco_unitario, status, regras_violadas
FROM ecommerce_lakehouse.silver.pedidos_quarentena
LIMIT 10;

## 06a · Clientes com histórico (SCD2)

Lista os clientes que têm mais de uma versão, mostrando o início e o fim de cada uma. A versão atual tem `__END_AT` nulo.

In [ ]:
%sql
SELECT cliente_id, email, telefone, __START_AT, __END_AT
FROM ecommerce_lakehouse.silver.clientes
WHERE cliente_id IN (
  SELECT cliente_id FROM ecommerce_lakehouse.silver.clientes
  GROUP BY cliente_id HAVING count(*) > 1
)
ORDER BY cliente_id, __START_AT
LIMIT 20;

## 06b · Versões vigentes e históricas

Conta as versões atuais e as antigas da tabela de clientes e o total de clientes distintos.

In [ ]:
%sql
SELECT count(*) FILTER (WHERE __END_AT IS NULL)     AS versoes_vigentes,
       count(*) FILTER (WHERE __END_AT IS NOT NULL) AS versoes_historicas,
       count(DISTINCT cliente_id)                   AS clientes_distintos
FROM ecommerce_lakehouse.silver.clientes;

## 06c · Pedidos sem duplicidade (SCD1)

Compara o número de linhas com o de pedidos distintos; os dois valores devem ser iguais, pois cada pedido guarda só o estado atual.

In [ ]:
%sql
SELECT count(*) AS linhas, count(DISTINCT pedido_id) AS ids
FROM ecommerce_lakehouse.silver.pedidos;

## 06d · Pedidos por status

Conta os pedidos de cada status atual, já com as mudanças do segundo lote aplicadas.

In [ ]:
%sql
SELECT status, count(*) AS qtd
FROM ecommerce_lakehouse.silver.pedidos
GROUP BY status
ORDER BY qtd DESC;

## 06e · Produtos após o delete

Compara os produtos distintos da bronze com os da silver; a diferença corresponde aos produtos removidos pelo CDC.

In [ ]:
%sql
SELECT (SELECT count(DISTINCT produto_id) FROM ecommerce_lakehouse.bronze.produtos_raw) AS produtos_na_bronze,
       (SELECT count(*) FROM ecommerce_lakehouse.silver.produtos)                        AS produtos_na_silver;

## 06f · Histórico da tabela de pedidos

Lista as versões da tabela `silver.pedidos`, com a operação e as métricas de cada uma.

In [ ]:
%sql
SELECT version, timestamp, operation, operationMetrics
FROM (DESCRIBE HISTORY ecommerce_lakehouse.silver.pedidos)
ORDER BY version;

## 06g · Change Data Feed

Conta as mudanças registradas no Change Data Feed por tipo. Se der erro, troque o número da versão por uma que exista no histórico da seção anterior.

In [ ]:
%sql
SELECT _change_type, count(*) AS qtd
FROM table_changes('ecommerce_lakehouse.silver.pedidos', 1)
GROUP BY _change_type;

## 07a · Permissões no catálogo

Lista quem tem permissão no catálogo, incluindo os grupos de conta do projeto.

In [ ]:
%sql
SHOW GRANTS ON CATALOG ecommerce_lakehouse;

## 07b · Schemas do catálogo

Lista os schemas do catálogo com o dono e o comentário de cada um.

In [ ]:
%sql
SELECT schema_name, schema_owner, comment
FROM ecommerce_lakehouse.information_schema.schemata
WHERE schema_name <> 'information_schema'
ORDER BY schema_name;

## 07c · Tags de PII

Mostra as tags aplicadas aos schemas, que indicam quais camadas contêm dados pessoais.

In [ ]:
%sql
SELECT * FROM ecommerce_lakehouse.information_schema.schema_tags;

## 07d · Linhagem entre tabelas

Mostra, para cada tabela de destino, de quais tabelas de origem ela é alimentada. Se vier vazia, use a aba Lineage no Catalog Explorer.

In [ ]:
%sql
SELECT source_table_schema, source_table_name,
       target_table_schema, target_table_name,
       max(event_time) AS ultimo_evento
FROM system.access.table_lineage
WHERE target_table_catalog = 'ecommerce_lakehouse'
  AND source_table_catalog = 'ecommerce_lakehouse'
GROUP BY ALL
ORDER BY target_table_schema, target_table_name;

## 08a · Máscaras de coluna

Lista as colunas da camada gold que têm uma função de máscara associada e qual é a função.

In [ ]:
%sql
SELECT * FROM ecommerce_lakehouse.information_schema.column_masks
WHERE table_schema = 'gold';

## 08b · Filtros de linha

Lista os filtros de linha aplicados às tabelas da gold. Vem vazio porque o filtro por UF foi implementado na view regional (seção 08h).

In [ ]:
%sql
SELECT * FROM ecommerce_lakehouse.information_schema.row_filters
WHERE table_schema = 'gold';

## 08c · Permissões na gold

Lista quem pode acessar o schema gold e com quais privilégios.

In [ ]:
%sql
SHOW GRANTS ON SCHEMA ecommerce_lakehouse.gold;

## 08d · Permissões na silver

Lista quem pode acessar o schema silver. Analistas e leitores de PII não devem aparecer aqui.

In [ ]:
%sql
SHOW GRANTS ON SCHEMA ecommerce_lakehouse.silver;

## 08e · Permissões na bronze

Lista quem pode acessar o schema bronze. Analistas e leitores de PII não devem aparecer aqui.

In [ ]:
%sql
SHOW GRANTS ON SCHEMA ecommerce_lakehouse.bronze;

## 08f · Consulta na tabela com máscara

Consulta a dimensão de clientes. O resultado depende do grupo de quem executa: membros de `ecom_engenharia` e `ecom_pii_leitores` veem os valores reais, os demais veem os valores mascarados.

In [ ]:
%sql
SELECT cliente_id, nome, email, cpf, telefone, uf
FROM ecommerce_lakehouse.gold.dim_clientes
LIMIT 10;

## 08g · Simulação do formato mascarado

Aplica a lógica das funções de máscara sem a checagem de grupo, só para mostrar o formato do dado mascarado. É uma simulação, não o resultado visto por um analista.

In [ ]:
%sql
SELECT concat('***@', split(email, '@')[1])  AS email_mascarado,
       concat('***.***.*', right(cpf, 2))    AS cpf_mascarado,
       concat('****', right(telefone, 4))    AS telefone_mascarado
FROM ecommerce_lakehouse.gold.dim_clientes
LIMIT 10;

## 08h · View regional

Conta os clientes por UF na view que aplica a máscara e o filtro de UF por cima da dimensão de clientes.

In [ ]:
%sql
SELECT uf, count(*) AS clientes
FROM ecommerce_lakehouse.gold.vw_clientes_regional
GROUP BY uf
ORDER BY uf;

## 09a · Execução do Z-Order

Mostra as operações `OPTIMIZE` da tabela de auditoria, com as colunas usadas no Z-Order e a quantidade de arquivos removidos e criados na compactação.

In [ ]:
%sql
SELECT version, timestamp, operation,
       operationParameters['zOrderBy']     AS zorder_por,
       operationMetrics['numRemovedFiles'] AS arquivos_removidos,
       operationMetrics['numAddedFiles']   AS arquivos_criados
FROM (DESCRIBE HISTORY ecommerce_lakehouse.auditoria.pedidos_snapshot)
WHERE operation = 'OPTIMIZE';

## 09b · Propriedades de silver.clientes

Mostra as propriedades da tabela, como os vetores de exclusão, o Change Data Feed e as colunas de Z-Order configuradas.

In [ ]:
%sql
SHOW TBLPROPERTIES ecommerce_lakehouse.silver.clientes;

## 09c · Propriedades de silver.pedidos

Mostra as mesmas propriedades para a tabela de pedidos, cujo Z-Order usa `cliente_id` e `pedido_ts`.

In [ ]:
%sql
SHOW TBLPROPERTIES ecommerce_lakehouse.silver.pedidos;

## 09d · Liquid clustering na gold

Exibe os metadados de uma materialized view da gold: as colunas de clustering (`# Clustering Information`), os vetores de exclusão, as versões de protocolo do Delta e o status da última atualização.

In [ ]:
%sql
DESCRIBE EXTENDED ecommerce_lakehouse.gold.vendas_diarias_categoria;

## 10 · Recursos de computação do pipeline

Mostra o uso de recursos da computação serverless registrado pelo pipeline no event log.

In [ ]:
%sql
SELECT timestamp, details:cluster_resources AS cluster_resources
FROM ecommerce_lakehouse.monitoramento.vw_event_log
WHERE event_type = 'cluster_resources'
ORDER BY timestamp DESC
LIMIT 20;

## 11a · Operações de escrita e restauração

Lista as operações de escrita, atualização, exclusão e restauração (`RESTORE`) da tabela de auditoria.

In [ ]:
%sql
SELECT version, timestamp, operation, operationParameters
FROM (DESCRIBE HISTORY ecommerce_lakehouse.auditoria.pedidos_snapshot)
WHERE operation IN ('WRITE', 'CREATE TABLE AS SELECT', 'UPDATE', 'DELETE', 'MERGE', 'RESTORE')
ORDER BY version;

## 11b · Time travel

Compara a contagem de linhas da versão 0 da tabela com a da versão atual, lendo a versão antiga com `VERSION AS OF`.

In [ ]:
%sql
SELECT (SELECT count(*) FROM ecommerce_lakehouse.auditoria.pedidos_snapshot VERSION AS OF 0) AS linhas_versao_0,
       (SELECT count(*) FROM ecommerce_lakehouse.auditoria.pedidos_snapshot)                  AS linhas_atual;

## 12a · Checkpoint do Auto Loader

Lista os arquivos de checkpoint e de schema do Auto Loader standalone, que permitem retomar o processamento de onde parou.

In [ ]:
%sql
LIST '/Volumes/ecommerce_lakehouse/auditoria/checkpoints/demo_autoloader_pedidos';

## 12b · Deduplicação com watermark

Compara as linhas da tabela do demo com os pedidos distintos, mostrando o efeito do `dropDuplicatesWithinWatermark`.

In [ ]:
%sql
SELECT count(*) AS linhas, count(DISTINCT pedido_id) AS pedidos_distintos
FROM ecommerce_lakehouse.auditoria.demo_autoloader_pedidos;

## 12c · Pedidos por hora

Mostra a agregação por janela de uma hora e canal, calculada em streaming com watermark de duas horas. As janelas mais recentes só aparecem depois que chegam eventos mais novos.

In [ ]:
%sql
SELECT janela, canal, qtd_pedidos, receita_estimada
FROM ecommerce_lakehouse.gold.pedidos_por_hora
ORDER BY janela DESC
LIMIT 20;

## 13a · Tabela de categorias

Mostra a tabela de referência de categorias, mantida fora do pipeline e carregada por `MERGE`.

In [ ]:
%sql
SELECT * FROM ecommerce_lakehouse.referencia.categorias ORDER BY categoria_id;

## 13b · Constraints da tabela de categorias

Lista as constraints definidas no schema de referência, como a chave primária e a regra de margem.

In [ ]:
%sql
SELECT * FROM ecommerce_lakehouse.information_schema.table_constraints
WHERE table_schema = 'referencia';

## 13c · Receita por categoria

Soma a receita e os pedidos por categoria na gold, mostrando que a tabela de referência enriquece as vendas.

In [ ]:
%sql
SELECT nome_categoria, sum(receita_dia) AS receita_total, sum(qtd_pedidos) AS pedidos
FROM ecommerce_lakehouse.gold.vendas_diarias_categoria
GROUP BY nome_categoria
ORDER BY receita_total DESC;

## 14 · Manutenção da tabela

Lista as operações de manutenção (`OPTIMIZE`, `VACUUM` e `REORG`) registradas no histórico da tabela de auditoria.

In [ ]:
%sql
SELECT version, timestamp, operation, operationParameters, operationMetrics
FROM (DESCRIBE HISTORY ecommerce_lakehouse.auditoria.pedidos_snapshot)
WHERE operation LIKE 'OPTIMIZE%' OR operation LIKE 'VACUUM%' OR operation LIKE 'REORG%'
ORDER BY version;

## 15a · Linhas processadas e descartadas

Soma, por flow do pipeline, as linhas processadas e as descartadas pelas expectations.

In [ ]:
%sql
SELECT origin.flow_name AS flow,
       sum(details:flow_progress.metrics.num_output_rows::BIGINT) AS linhas_processadas,
       sum(coalesce(details:flow_progress.data_quality.dropped_records::BIGINT, 0)) AS linhas_descartadas
FROM ecommerce_lakehouse.monitoramento.vw_event_log
WHERE event_type = 'flow_progress'
GROUP BY ALL
ORDER BY flow;

## 15b · Duração das atualizações

Mostra o início, o fim e a duração em segundos de cada atualização do pipeline.

In [ ]:
%sql
SELECT origin.update_id AS update_id,
       min(timestamp)   AS inicio,
       max(timestamp)   AS fim,
       cast(max(timestamp) AS DOUBLE) - cast(min(timestamp) AS DOUBLE) AS duracao_segundos
FROM ecommerce_lakehouse.monitoramento.vw_event_log
WHERE event_type = 'update_progress'
GROUP BY origin.update_id
ORDER BY inicio DESC;

## 15c · Planejamento das materialized views

Mostra as informações de planejamento do pipeline, que indicam como cada materialized view foi atualizada.

In [ ]:
%sql
SELECT timestamp, message, details:planning_information AS planejamento
FROM ecommerce_lakehouse.monitoramento.vw_event_log
WHERE event_type = 'planning_information'
ORDER BY timestamp DESC
LIMIT 20;